##If your IDE doesn't add a venv or .venv automatically:
python -m venv venv

source venv/Scripts/activate (Windows: venv/Scripts/activate)

pip install -r requirements.txt

pip freeze > requirements.txt

##After installing Docker Desktop:
docker compose up -d

In [1]:
import pandas as pd
from pymongo import MongoClient

In [3]:
def ingest_csv(csv_path, db_name, collection_name, sep=';', encoding='utf-8'):
    client = MongoClient("mongodb://localhost:27017/")
    db = client[db_name]
    collection = db[collection_name]

    try:
        df = pd.read_csv(
            csv_path,
            sep=sep,
            dtype=str,
            encoding=encoding,
            keep_default_na=False,
        )

        if df.empty:
            print("The CSV file is empty.")
            return

        # Data Cleaning
        # MongoDB does not accept float 'NaN'. Also convert empty strings to None (null).
        df = df.where(pd.notnull(df), None)
        df = df.replace({"": None})

        # Convert to list of dictionaries
        data_dict = df.to_dict('records')

        # Insert into MongoDB
        if data_dict:
            collection.drop()
            result = collection.insert_many(data_dict)
            print(f"Successfully inserted {len(result.inserted_ids)} records from {csv_path}.")
        else:
            print("File is empty.")

    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        client.close()

In [4]:
ingest_csv(
        csv_path="raw_data/birthrate/OGD_indquot001_HVD_INDQUOTE_3.csv", 
        db_name="mongo_db", 
        collection_name="raw_birthrate"
    )

Successfully inserted 1104 records from raw_data/birthrate/OGD_indquot001_HVD_INDQUOTE_3.csv.


In [5]:
ingest_csv(
        csv_path="raw_data/fertility/OGD_indquot001_HVD_INDQUOTE_6_C-DEMIND_NUTS3-0.csv", 
        db_name="mongo_db", 
        collection_name="raw_fertility_rate"
    )

Successfully inserted 48 records from raw_data/fertility/OGD_indquot001_HVD_INDQUOTE_6_C-DEMIND_NUTS3-0.csv.


In [6]:
def ingest_ods(file_path, db_name, collection_name, sheet_name=None, drop_rows=None):
    client = MongoClient("mongodb://localhost:27017/")
    db = client[db_name]
    collection = db[collection_name]

    try:
        resolved_sheet = 0 if sheet_name is None else sheet_name
        df = pd.read_excel(
            file_path,
            engine='odf',
            header=None,
            sheet_name=resolved_sheet,
        )

        if drop_rows is None:
            drop_rows_resolved = []
        elif isinstance(drop_rows, int):
            drop_rows_resolved = [drop_rows]
        else:
            drop_rows_resolved = list(drop_rows)

        # Drop selected rows (0-based indices) before header normalization
        if drop_rows_resolved:
            df = df.drop(index=drop_rows_resolved, errors='ignore').reset_index(drop=True)

        # Use the (remaining) first row as header
        header = df.iloc[0].fillna("").astype(str).str.strip()
        df = df.iloc[1:].copy()
        df.columns = header

        # Drop the last two data rows
        df = df.iloc[:-2].copy().reset_index(drop=True)

        # Data Cleaning
        # MongoDB does not accept float 'NaN'. Convert them to None (null).
        df = df.where(pd.notnull(df), None)

        # Convert DataFrame to dictionary records
        data_dict = df.to_dict("records")

        # Insert into MongoDB
        if data_dict:
            collection.drop()
            result = collection.insert_many(data_dict)
            print(f"Successfully inserted {len(result.inserted_ids)} records from {file_path}.")
        else:
            print("File is empty.")

    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        client.close()

In [7]:
ingest_ods(
        file_path="raw_data/Gross_Regional_Product_NUTS3.ods", 
        db_name="mongo_db", 
        collection_name="raw_GRP", 
        sheet_name="GRP",
        drop_rows=(0, 2)
    )

Successfully inserted 36 records from raw_data/Gross_Regional_Product_NUTS3.ods.


In [8]:
ingest_ods(
        file_path="raw_data/Gross_Regional_Product_NUTS3.ods", 
        db_name="mongo_db", 
        collection_name="raw_GRP_per_capita", 
        sheet_name="GRP_per_capita",
        drop_rows=(0, 2)
    )

Successfully inserted 36 records from raw_data/Gross_Regional_Product_NUTS3.ods.


In [9]:
ingest_ods(
        file_path="raw_data/HPIMesszahlen.ods", 
        db_name="mongo_db", 
        collection_name="raw_HPI", 
        sheet_name=None,
        drop_rows=(0, 2)
    )

Successfully inserted 79 records from raw_data/HPIMesszahlen.ods.


In [10]:
ingest_ods(
        file_path="raw_data/Tab2_erwerbstaetigenquoten_nach_alter_und_geschlecht_seit_1994.ods", 
        db_name="mongo_db", 
        collection_name="raw_employment_rate", 
        sheet_name=None,
        drop_rows=(0, 3)
    )

Successfully inserted 95 records from raw_data/Tab2_erwerbstaetigenquoten_nach_alter_und_geschlecht_seit_1994.ods.


C:\Users\elias\AppData\Local\Temp\ipykernel_23700\2594476412.py:39: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  data_dict = df.to_dict("records")
